# 第10講　実習：生データの処理
データベースから入手した生データを、分析に使える形に整える下ごしらえを練習する。
列の選択・改名・部分集合・値の置き換え、日付データの処理、文字列データの処理を扱う。

## 0 準備

In [ ]:
import pandas as pd
import numpy as np
import pyodide

## 1 データ処理の基礎（IRIS）

In [ ]:
data = pd.read_csv(pyodide.http.open_url("https://raw.githubusercontent.com/ChungWookyung/kg-jupyterlite-data-analysis/main/content/DATA/iris.csv"))
data.head()

IRIS（アヤメ）データを読み込んだ。列は sepal.length / sepal.width / petal.length / petal.width / variety の5つ。

### 1-1 行と列の概念

In [ ]:
data.columns

In [ ]:
data.shape

`columns` は列名の一覧、`shape` はデータの形 `(行数, 列数)` を返す。ここでは 150行×5列。

### 1-2 列を取り出す

In [ ]:
data["sepal.width"]

In [ ]:
data[["petal.length", "sepal.length"]]

`data["列名"]` で1列（Series）、`data[["列名1","列名2"]]` とリストで囲むと複数列を指定した順で取り出せる。

### 1-3 列名を変更する

In [ ]:
data2 = data.copy()
data2.columns = ["がく片の長さ", "がく片の広さ", "花びらの長さ", "花びらの広さ", "種類"]
data2.head()

`copy()` で複製してから、`data2.columns` に列数と同じ長さのリストを代入すると列名を丸ごと差し替えられる。

### 1-4 データの部分集合（iloc）

In [ ]:
data.iloc[0, :]

In [ ]:
data.iloc[:, 2]

`iloc[行, 列]` は番号（0から）で部分集合を取り出す。`:` は「全部」。範囲は `iloc[2:5, 1:4]` のように指定できる（終了は含まない）。

### 1-5 列を作る・置き換える

In [ ]:
data2 = data.copy()
data2["NEW"] = 1
data2["sepal.width"] = -100
data2.head()

存在しない列名に代入すると新しい列が作られ、存在する列名に代入すると中身が置き換わる。

### 1-6 列に計算・関数を適用する

In [ ]:
data2 = data.copy()
data2["petal.width100"] = data2["petal.width"] * 100
data2["sepal.length_log"] = np.log(data2["sepal.length"])
data2.head()

列への四則演算や `np.log()` などの関数は全ての行にまとめて適用される。結果を新しい列名に代入すれば保存できる。

### 1-7 dict（辞書）と値の置き換え（replace）

In [ ]:
d = {"あ": "A", "い": "I", "う": "U", "え": "E", "お": "O"}
d["い"]

dict はキー（左）とバリュー（右）を一対一で対応させるデータ型。`replace()` はこの dict を使って値をまとめて置き換える。

In [ ]:
data["variety"].unique()

In [ ]:
data2 = data.copy()
data2["variety"] = data2["variety"].replace(
    {"Setosa": "せとさ", "Versicolor": "ばーしからー", "Virginica": "ばーじにか"})
data2.head()

`unique()` で値を確認し、`replace({旧:新})` で置き換える。置き換えた列を元の列に代入すれば変換結果が保存される。

## 実習①　データ処理の基礎（IRIS）
1. `variety` 列だけを取り出してください。
2. 列の順番を `variety, sepal.width, petal.width, sepal.length, petal.length` に並べ替えてください。
3. `data` を `data2` にコピーし、列名を `["A","B","C","D","E"]` に変えてください。
4. `iloc` を使って、3〜5行目・2〜5列目を取り出してください。
5. `data` を `data2` にコピーし、新しい列 `"A"` を追加して、全ての値を文字列 `"abcd"` にしてください。
6. `data` を `data2` にコピーし、`sepal.width` 列に `np.sin()` を適用して `"sepal.width_sin"` 列を追加してください。
7. `data` を `data2` にコピーし、`variety` を `Setosa`→`"S"`、`Versicolor`→`"C"`、`Virginica`→`"V"` に置き換えてください。

## 2 実習：CPSデータ

In [ ]:
data = pd.read_csv(pyodide.http.open_url("https://raw.githubusercontent.com/ChungWookyung/kg-jupyterlite-data-analysis/main/content/DATA/CPS1988.csv"))
data.head()

In [ ]:
data["region"].unique()

CPS＝米国 Current Population Survey（1988年）。約28,000人分の労働データ。列は wage（賃金）/ education（学歴）/ experience（経験年数）/ ethnicity（人種）/ smsa（都心部か）/ region（地域）/ parttime（パートか）。

## 実習②　CPSデータ
1. `data` を `data2` にコピーし、列名を日本語に置き換えてください（`smsa` は「都心部」と訳す）。
2. `data` を `data2` にコピーし、`wage` 列に `np.log()` を適用して `"wage_log"` 列を追加してください。
3. `data` を `data2` にコピーし、`ethnicity` の `cauc` を「白人」、`afam` を「黒人」に置き換えてください。
4. `data` を `data2` にコピーし、`smsa` 列と `parttime` 列の値を、`yes`→「はい」・`no`→「いいえ」に置き換えてください。
5. `data` を `data2` にコピーし、`region` 列の値（`northeast` など）を日本語に翻訳して置き換えてください。

## 3 時間軸の処理

### 3-1 日付を作る（date_range）

In [ ]:
pd.date_range(start="2000/01/01", end="2030/01/01")

In [ ]:
pd.date_range(start="2000/01/01", periods=365)

`date_range()` は連続した日付（Datetime）を作る。`start`+`end` で毎日、`start`+`periods` で個数分の日付が得られる。

### 3-2 日付の間隔（freq）

In [ ]:
pd.date_range(start="2000/01/01", periods=12, freq="ME")

In [ ]:
pd.date_range(start="2000/01/01", periods=10, freq="YE")

In [ ]:
pd.date_range(start="2000/01/01", periods=12, freq="MS")

In [ ]:
pd.date_range(start="2000/01/01", periods=10, freq="YS")

`freq` で間隔を指定する。`ME`＝月末、`YE`＝年末、`MS`＝月初（各月1日）、`YS`＝年初（各年1月1日）。以前の `"M"` `"Y"` は pandas 2.2 以降で廃止予定なので `"ME"` `"YE"` を使う。

### 3-3 年・月・日を取り出す

In [ ]:
d = pd.date_range(start="2000/01/01", end="2030/01/01")
d.year

In [ ]:
d.month

In [ ]:
d.day

Datetime の並びには `year` `month` `day` の属性があり、年・月・日を数値で取り出せる。

### 3-4 データの列から年月日を取り出す

In [ ]:
data = pd.DataFrame({"Date": d, "Value": 1})
data.head()

In [ ]:
pd.DatetimeIndex(data["Date"]).year

In [ ]:
data["Year"] = pd.DatetimeIndex(data["Date"]).year
data.head()

DataFrame の列から年月日を取るには、まず `pd.DatetimeIndex(列)` を通してから `.year` などを付ける。新しい列に代入して保存できる。

### 3-5 日付を文字列にする（strftime）

In [ ]:
pd.DatetimeIndex(data["Date"]).strftime("%Y年%m月%d日")

In [ ]:
data["JPN_Date"] = pd.DatetimeIndex(data["Date"]).strftime("%Y年%m月%d日")
data.head()

`strftime()` は日付を指定した書式の文字列に変換する。`%Y`＝年、`%m`＝月、`%d`＝日。

## 実習③　時間軸の処理
1. 2010年6月1日から今日まで、毎日の日付を生成してください。
2. 自分の誕生日から20年間、毎月の日付を生成してください（`freq="ME"`）。
3. 1990年3月1日から今年まで、毎年1月1日の日付を生成してください（`freq="YS"`）。
4. `data` に `"Month"` 列を追加し、`Date` 列の月を入れてください。
5. `data` に `"Day"` 列を追加し、`Date` 列の日を入れてください。
6. `data` に `"Date_New"` 列を追加し、日付を日/月/年の文字列にして入れてください（例：2022年6月13日 → `"13/06/2022"`。ヒント：`strftime("%d/%m/%Y")`）。

## 4 実習：株価指数データ

In [ ]:
data = pd.read_csv(pyodide.http.open_url("https://raw.githubusercontent.com/ChungWookyung/kg-jupyterlite-data-analysis/main/content/DATA/nikkei.csv"))
data["DATE"] = pd.DatetimeIndex(data["DATE"])
data.head()

日経平均株価（NIKKEI225）の日次データ。列は DATE（日付）と NIKKEI225（終値）。読み込んだ直後に DATE を日付型へ変換しておく。

In [ ]:
data["WEEKDAY"] = pd.DatetimeIndex(data["DATE"]).weekday
data.groupby("WEEKDAY")["NIKKEI225"].mean()

`weekday` は曜日（月=0 … 日=6）。`groupby("列")["対象列"].mean()` で、指定した列の値ごとにグループの平均を求める。

## 実習④　株価指数データ
1. `YEAR` `MONTH` `DAY` 列を追加し、`DATE` 列の年・月・日を入れてください。
2. `DATE_JPN` 列を追加し、日本語表記（`○年○月○日`）の日付を入れてください。
3. `groupby()` を使って、`NIKKEI225` の年別平均と月別平均を求めてください（ヒント：`data.groupby("YEAR")["NIKKEI225"].mean()`）。

## 5 文字列の処理

### 5-1 文字列の一括処理

In [ ]:
data = pd.DataFrame({"col1": ["A","A","A","A","A"], "col2": "B"})
data.head()

In [ ]:
data["col1"] + "1"

In [ ]:
"a" + data["col1"]

In [ ]:
data["col1"] + data["col2"]

文字列の列は `+` で連結でき、全ての行にまとめて適用される。「列＋文字」「文字＋列」「列＋列」のどれも同じように書ける。

In [ ]:
data2 = data.copy()
data2["col3"] = data2["col1"] + "1"
data2["col4"] = "a" + data2["col1"]
data2["col5"] = data2["col1"] + data2["col2"]
data2.head()

連結した結果を新しい列名に代入すれば、加工後の文字列を列として残せる。

### 5-2 str：文字列の一部を取り出す

In [ ]:
data = pd.read_csv(pyodide.http.open_url("https://raw.githubusercontent.com/ChungWookyung/kg-jupyterlite-data-analysis/main/content/DATA/nikkei.csv"))
data.head()

In [ ]:
data["DATE"].str[0:4]

In [ ]:
data["DATE"].str[5:7]

In [ ]:
data["DATE"].str[8:10]

`DATE` を文字列のまま読み込み、`.str[開始:終了]` で各行の部分文字列を取り出す。`[0:4]` は年、`[5:7]` は月、`[8:10]` は日。

### 5-3 split()：区切り文字で分解する

In [ ]:
d = data["DATE"].str.split("-", expand=True)
d.head()

In [ ]:
d[0].head()

In [ ]:
d[1].head()

In [ ]:
d[2].head()

`str.split("区切り", expand=True)` は区切り文字で文字列を分解し、複数の列に分ける。`d[0]` `d[1]` `d[2]` で各部分を取り出せる。

## 実習⑤　文字列の処理
1. `data` を `data2` にコピーし、文字列の一括処理を使った新しい列を3つ追加してください（例：`col1 + "1"`、`"a" + col1`、`col1 + col2` などを自由に組み合わせる）。
2. `.str[開始:終了]` を使って、`YEAR` `MONTH` `DAY` の3列を追加してください（値は年・月・日）。
3. `.str.split("-", expand=True)` を使って、同じく `YEAR` `MONTH` `DAY` の3列を追加してください。